# F2_01 · Obtención del subconjunto de La Araucanía

**Objetivo:** Generar, a partir de la base de datos nacional chilena de egresos hospitalarios del año 2022 del DEIS, un subconjunto con los egresos de personas residentes de la Región de La Araucanía, para usarlos en los notebooks siguientes.

**Fuente:** Egresos hospitalarios 2022, DEIS, Ministerio de Salud de Chile. https://repositoriodeis.minsal.cl/DatosAbiertos/EGRESOS/EGRESOS_2022.zip

**Entrada:** data/raw/EGRE_DATOS_ABIERTOS_2022.csv (base nacional, no se sube a GitHub)

**Salida:** data/processed/egresos_araucania_2022.csv

**Qué NO hace este notebook:** No filtra los diagnósticos CIE-10 K80.0–K80.8, ni determina los grupos de riesgo, la provincia ni si tuvo o no una colecistectomía; eso se realizará en el notebook de transformación (F2_03).

In [1]:
import pandas as pd
from pathlib import Path

# El notebook está en F2, así que la raíz del proyecto es la carpeta de arriba
RAIZ = Path.cwd()
if RAIZ.name.startswith("F"):
    RAIZ = RAIZ.parent

ENTRADA = RAIZ / "data" / "raw" / "EGRE_DATOS_ABIERTOS_2022.csv"
SALIDA = RAIZ / "data" / "processed" / "egresos_araucania_2022.csv"

print("¿Existe la entrada?", ENTRADA.exists())
print(ENTRADA)

¿Existe la entrada? True
C:\Users\Paula Becerra\Documents\magister\proyecto-grupo2-mcdi500\data\raw\EGRE_DATOS_ABIERTOS_2022.csv


In [2]:
egresos = pd.read_csv(
    ENTRADA,
    sep=";",
    encoding="latin-1",
    dtype=str
)

print("Base nacional leída")

Base nacional leída


In [3]:
print("Filas y columnas:", egresos.shape)
print()
print(egresos.columns.tolist())
egresos.head()


Filas y columnas: (1597118, 18)

['PERTENENCIA_ESTABLECIMIENTO_SALUD', 'SEXO', 'GRUPO_EDAD', 'ETNIA', 'GLOSA_PAIS_ORIGEN', 'COMUNA_RESIDENCIA', 'GLOSA_COMUNA_RESIDENCIA', 'REGION_RESIDENCIA', 'GLOSA_REGION_RESIDENCIA', 'PREVISION', 'GLOSA_PREVISION', 'ANO_EGRESO', 'DIAG1', 'DIAG2', 'DIAS_ESTADA', 'CONDICION_EGRESO', 'GLOSA_INTERV_Q_PPAL', 'GLOSA_PROCED_PPAL']


,PERTENENCIA_ESTABLECIMIENTO_SALUD,SEXO,GRUPO_EDAD,ETNIA,GLOSA_PAIS_ORIGEN,COMUNA_RESIDENCIA,GLOSA_COMUNA_RESIDENCIA,REGION_RESIDENCIA,GLOSA_REGION_RESIDENCIA,PREVISION,GLOSA_PREVISION,ANO_EGRESO,DIAG1,DIAG2,DIAS_ESTADA,CONDICION_EGRESO,GLOSA_INTERV_Q_PPAL,GLOSA_PROCED_PPAL
0,No Pertenecientes al Sistema Nacional de Servi...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,01101,Iquique,01,De Tarapacá,1,FONASA,2022,Z539,NaN,1,1,NaN,Sin Arancel FONASA
1,No Pertenecientes al Sistema Nacional de Servi...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,01101,Iquique,01,De Tarapacá,1,FONASA,2022,J209,NaN,2,1,NaN,NaN
2,No Pertenecientes al Sistema Nacional de Servi...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,01101,Iquique,01,De Tarapacá,1,FONASA,2022,J129,NaN,5,1,NaN,NaN
3,No Pertenecientes al Sistema Nacional de Servi...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,01101,Iquique,01,De Tarapacá,1,FONASA,2022,A099,NaN,2,1,NaN,NaN
4,No Pertenecientes al Sistema Nacional de Servi...,HOMBRE,1 a 9,No se identifica con alguna etnia,Chileno,01107,Alto Hospicio,01,De Tarapacá,1,FONASA,2022,Q531,NaN,1,1,Descenso testículo inguinal c/s hernioplastía,NaN


In [4]:
egresos[["REGION_RESIDENCIA", "GLOSA_REGION_RESIDENCIA"]].drop_duplicates().sort_values("REGION_RESIDENCIA")

,REGION_RESIDENCIA,GLOSA_REGION_RESIDENCIA
1557549,*,*
0,01,De Tarapacá
1001,02,De Antofagasta
3190,03,De Atacama
4061,04,De Coquimbo
6639,05,De Valparaíso
12730,06,Del Libertador B. O'Higgins
15684,07,Del Maule
19356,08,Del Bíobío
26516,09,De La Araucanía


In [5]:
es_araucania = egresos["REGION_RESIDENCIA"] == "09"
araucania = egresos[es_araucania]

print("Egresos de La Araucanía:", len(araucania))

Egresos de La Araucanía: 84618


In [6]:
print("Códigos de región que quedaron:", araucania["REGION_RESIDENCIA"].unique())
print("Nombres de región que quedaron:", araucania["GLOSA_REGION_RESIDENCIA"].unique())

no_09 = ~araucania["COMUNA_RESIDENCIA"].str.startswith("09")
print("Comunas que NO empiezan con 09:", no_09.sum())

sin_region = (egresos["REGION_RESIDENCIA"] == "*").sum()
print("Egresos del país sin región (*):", sin_region)

porcentaje = len(araucania) / len(egresos) * 100
print("La Araucanía es el", round(porcentaje, 1), "% de los egresos del país")

Códigos de región que quedaron: <StringArray>
['09']
Length: 1, dtype: str
Nombres de región que quedaron: <StringArray>
['De La Araucanía']
Length: 1, dtype: str
Comunas que NO empiezan con 09: 0
Egresos del país sin región (*): 39569
La Araucanía es el 5.3 % de los egresos del país


In [7]:
araucania.to_csv(SALIDA, index=False, encoding="utf-8")

print("Guardado en:", SALIDA)
print("Tamaño del archivo (MB):", round(SALIDA.stat().st_size / 1_000_000, 1))

Guardado en: C:\Users\Paula Becerra\Documents\magister\proyecto-grupo2-mcdi500\data\processed\egresos_araucania_2022.csv
Tamaño del archivo (MB): 18.8


In [8]:
verificacion = pd.read_csv(SALIDA, dtype=str)

print("Filas y columnas del archivo guardado:", verificacion.shape)
print("¿Es idéntico a lo que filtramos?", verificacion.equals(araucania.reset_index(drop=True)))

Filas y columnas del archivo guardado: (84618, 18)
¿Es idéntico a lo que filtramos? True


## Resultado

- Base nacional leída: 1597118 filas y 18 columnas.
- Egresos de residentes de La Araucanía: 84618 (5.3 % del país).
- Se excluyeron 39569 egresos con región desconocida (`*`), lo que puede generar un subconteo.
- Subconjunto guardado en `data/processed/egresos_araucania_2022.csv` (18.8 MB) y verificado al volver a leerlo.

**Para reproducir:** descargar la base desde el enlace del DEIS, descomprimirla en `data/raw/` y ejecutar este notebook de principio a fin.